In [1]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib ipympl

échauffement : le broadcasting

In [51]:
a = np.array([1, 2, 3])

# Question 1

print(a[:,None].shape)
print(a[None,:].shape)

(3, 1)
(1, 3)


In [ ]:
# Question 2

print((a[:,None]-a[None,:]).shape)
print(a[:,None]-a[None,:])
# l'élément [i,j] contient a[j]-a[i]

(3, 3)
[[ 0 -1 -2]
 [ 1  0 -1]
 [ 2  1  0]]


In [ ]:
# Question 3

b = np.arange(6).reshape((2,3))
print(b)
print((b[:, None, :] - b[:, :, None]).shape)
print(b[:, None, :] - b[:, :, None])
# l'élément [i,j,k] contient  

[[0 1 2]
 [3 4 5]]
(2, 3, 3)
[[[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]

 [[ 0  1  2]
  [-1  0  1]
  [-2 -1  0]]]


initialisation aléatoire

In [37]:
mass_max = 3.
    
x_min, x_max = -10., 10.
y_min, y_max = -10., 10.

speed_max = 1.

def init_problem(N):
    """
    retourne un tuple masses, positions, speeds
    de formes resp.   (N,)    (2, N)     (2, N)
    tiré au sort dans les bornes définies ci-dessus
    """
    masses = np.random.uniform(0.00000001,mass_max,size=(N,))
    positions = np.random.uniform(np.array([[x_min],[y_min]]),np.array([[x_max],[y_max]]),size=(2,N))
    speeds = np.random.uniform(-speed_max,speed_max,size=(2,N))
    return masses, positions, speeds

# pour tester

# normalement vous devez pouvoir faire ceci

masses, positions, speeds = init_problem(10)

# et ceci devrait afficher OK
try:
    assert masses.shape == (10,) and positions.shape == speeds.shape == (2, 10)
    print("OK")
except:
    print("KO")

OK


initialisation reproductible

In [28]:
# for your convenience

def init3():
    # first element is sun-like: heavy, at the center, and no speed
    masses = np.array([3, 1, 1], dtype=float)
    positions = np.array([
        [0, 5, -5], 
        [0, 1, -1]], dtype=float)
    speeds = np.array([
        [0, -1, 1], 
        [0, 0, 0]], dtype=float)
    return masses, positions, speeds


les forces

In [ ]:

def forces(masses, positions, G=1.0):
    """
    returns an array of shape (2,  N)
    that contains the force felt by each mass from all the others
    G is the Newton constant and by default is set to 1 
    (we have abstract units anyway)
    """
    tab_1 = positions[:,None,:] - positions[:,:,None]
    # matrice des rj-ri de forme (2,N,N)
    tab_2 = np.sqrt(tab_1[0,:,:]**2 + tab_1[1,:,:]**2)
    # matrice des |rj-ri| de forme (N,N) (même résultat que tab_2 = np.linalg.norm(tab_1,axis=0) après test)
    np.fill_diagonal(tab_2,np.inf)
    tab_3 = G*masses[:,None]*masses[None,:]*tab_1/tab_2**3
    tab_4 = np.sum(tab_3, axis=2)
    return tab_4

# pour tester, voici les valeurs attendues avec la config prédéfinie

masses, positions, speeds = init3()

forces(masses, positions)

# should be true
# np.all(np.isclose(f, np.array([
#     [ 0.        , -0.12257258,  0.12257258],
#     [ 0.        , -0.02451452,  0.02451452]])))


array([[ 0.        , -0.12257258,  0.12257258],
       [ 0.        , -0.02451452,  0.02451452]])

le simulateur

In [ ]:
def simulate(masses, positions, speeds, dt=0.1, nb_steps=100):
    """
    should return the positions across time
    so an array of shape (nb_steps, 2, N)
    optional dt is the time step
    """
    dim,N = masses.shape
    traj = np.zeros((nb_steps,dim,N))
    for i in range(nb_steps) :
        traj[i] = positions
        forces = forces(masses, positions)
        position = position + speeds*dt
        speeds = speeds + forces/masses*dt
    return traj
